# Import Libraries and Data and setup paths and a reproducible pipeline

In [26]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths
DATA_PATH = Path("../data/processed/milan_telecom_agg_all.parquet")
RESULTS_EDA_DIR = Path("../results/eda")  # already used in EDA
RESULTS_PREP_DIR = Path("../results/preprocessing")
RESULTS_MODELS_DIR = Path("../results/models")

# Create dirs
RESULTS_PREP_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH.resolve())
print("RESULTS_PREP_DIR:", RESULTS_PREP_DIR.resolve())

DATA_PATH: C:\Users\Lu\OneDrive\ToU\chl_Clustering\data\processed\milan_telecom_agg_all.parquet
RESULTS_PREP_DIR: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing


In [3]:
#load data + integrity checks

df = pd.read_parquet(DATA_PATH)

# Ensure datetime exists
if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["time_interval"], unit="ms")

print("Shape:", df.shape)
print("Datetime range:", df["datetime"].min(), "→", df["datetime"].max())
print("Unique squares:", df["square_id"].nunique())
print("Unique intervals:", df["time_interval"].nunique())

# Quick key check (should be unique after aggregation)
dup_rate = df.duplicated(["square_id", "time_interval"]).mean()
print("Duplicate rate on (square_id,time_interval):", dup_rate)

Shape: (89245318, 10)
Datetime range: 2013-10-31 23:00:00 → 2014-01-01 22:50:00
Unique squares: 10000
Unique intervals: 8928
Duplicate rate on (square_id,time_interval): 0.0


In [4]:
# Create and save a holdout split of squares (for later stability testing)

square_ids = df["square_id"].unique()

train_squares, holdout_squares = train_test_split(
    square_ids, test_size=0.2, random_state=RANDOM_STATE
)

split_path = RESULTS_PREP_DIR / "square_id_split.npz"
np.savez(split_path, train_squares=train_squares, holdout_squares=holdout_squares)

print("Saved split:", split_path.resolve())
print("Train squares:", len(train_squares), "| Holdout squares:", len(holdout_squares))

Saved split: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\square_id_split.npz
Train squares: 8000 | Holdout squares: 2000


Now, we have a clean foundation and saved artifacts. This will be very useful later when we compare preprocessing/feature choices and want to see if results are stable.

# Feature selection

## Chosen Temporal Feature Representation: Hour-of-Day × Weekday/Weekend Profiles

For clustering Milan grid cells into interpretable *urban activity types*, we represent each grid cell by its **average activity profile across the day**, split into **weekday vs weekend** regimes.

### Why this choice fits our objective
- **Captures mobility-relevant rhythms:** commuting peaks, daytime commercial intensity, and nighttime/leisure patterns are primarily expressed as *within-day* structure and *weekday vs weekend* differences.
- **Balances detail and interpretability:** 10-minute raw data might be too granular/noisy for clustering, while simple averages lose the shape of daily behavior. Hourly profiles preserve the *shape* stakeholders care about.
- **Supports actionable narratives:** clusters can be described with intuitive labels (e.g., “commuter-oriented,” “nighttime/leisure,” “steady daytime,” “mixed-use”), and validated using profile plots and maps.
- **Controls for regime shifts:** splitting weekday/weekend avoids blending two different behavioral regimes into one average profile.

This representation will be the foundation for downstream feature selection (variance/correlation/PCA checks) and clustering model comparison.

### Build the clustering feature matrix (per square)

Next, we convert the raw 10-minute activity table into a **single feature vector per grid cell** (`square_id`).

We do this by:
1. Deriving **hour-of-day** and **weekday/weekend** flags from `datetime`.
2. Computing the **mean activity** in each hour bin, separately for weekdays and weekends.
3. Flattening these hourly profiles into a wide matrix where each row is one `square_id` and each column is an interpretable feature (e.g., `weekday_hour_08_call_out_mean`).

This creates a clustering-ready dataset that preserves daily rhythms while reducing noise and dimensionality relative to the original 10-minute resolution.

In [5]:
# create hourly weekday/weekend mean profiles per square

activity_cols = ["sms_in", "sms_out", "call_in", "call_out", "internet_traffic"]

# Ensure required time columns exist
df["hour"] = df["datetime"].dt.hour
df["weekday"] = df["datetime"].dt.weekday
df["is_weekend"] = df["weekday"].isin([5, 6])

# Map to a clean regime label
df["regime"] = np.where(df["is_weekend"], "weekend", "weekday")

# Group and compute mean profiles
grp = (
    df.groupby(["square_id", "regime", "hour"], as_index=False)[activity_cols]
      .mean()
)

# Pivot to wide format: columns become (regime, hour, channel)
wide = grp.pivot(index="square_id", columns=["regime", "hour"], values=activity_cols)

# Flatten column names
wide.columns = [f"{reg}_hour_{hour:02d}_{chan}_mean" for chan, reg, hour in wide.columns]
X = wide.reset_index()

print("Feature matrix shape:", X.shape)
X.head()

Feature matrix shape: (10000, 241)


,square_id,weekday_hour_00_sms_in_mean,weekday_hour_01_sms_in_mean,weekday_hour_02_sms_in_mean,weekday_hour_03_sms_in_mean,weekday_hour_04_sms_in_mean,weekday_hour_05_sms_in_mean,weekday_hour_06_sms_in_mean,weekday_hour_07_sms_in_mean,weekday_hour_08_sms_in_mean,...,weekend_hour_14_internet_traffic_mean,weekend_hour_15_internet_traffic_mean,weekend_hour_16_internet_traffic_mean,weekend_hour_17_internet_traffic_mean,weekend_hour_18_internet_traffic_mean,weekend_hour_19_internet_traffic_mean,weekend_hour_20_internet_traffic_mean,weekend_hour_21_internet_traffic_mean,weekend_hour_22_internet_traffic_mean,weekend_hour_23_internet_traffic_mean
0,1,0.190282,0.071327,0.040567,0.031687,0.039662,0.117830,0.389212,0.910665,1.037141,...,13.579558,13.585430,13.349345,13.925562,13.909820,14.167613,14.784274,13.456605,12.156795,9.718212
1,2,0.192204,0.071691,0.040877,0.032114,0.040092,0.119678,0.395245,0.924243,1.053721,...,13.641217,13.647976,13.403424,13.979457,13.970274,14.221566,14.832661,13.500848,12.193845,9.740116
2,3,0.194250,0.072079,0.041207,0.032568,0.040549,0.121644,0.401667,0.938696,1.071369,...,13.706851,13.714553,13.460988,14.036826,14.034625,14.278996,14.884168,13.547942,12.233283,9.763432
3,4,0.184715,0.070271,0.039669,0.030452,0.038418,0.112479,0.371735,0.871337,0.989119,...,13.400960,13.404264,13.192706,13.769455,13.734714,14.011339,14.644117,13.328457,12.049480,9.654766
4,5,0.167339,0.063908,0.036700,0.028211,0.035827,0.104840,0.347525,0.814332,0.923792,...,12.245734,12.221951,12.026093,12.512307,12.471926,12.727311,13.272163,12.084236,10.913208,8.751670


This produces a feature table with:

- 10,000 rows (squares)
- 5 channels × 24 hours × 2 regimes = 240 features

In [6]:
# Save the feature matrix for later use

out_path = RESULTS_PREP_DIR / "features_hourly_weekday_weekend_mean.parquet"
X.to_parquet(out_path, index=False)

print("Saved feature matrix:", out_path.resolve())

Saved feature matrix: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\features_hourly_weekday_weekend_mean.parquet


In [7]:
# Sanity check for missing intervals

missing_frac = X.isna().mean().sort_values(ascending=False)
print("Top missingness columns:")
display(missing_frac.head(10))

print("Rows with any missing:", (X.isna().any(axis=1)).mean())

Top missingness columns:


square_id                      0.0
weekday_hour_00_sms_in_mean    0.0
weekday_hour_01_sms_in_mean    0.0
weekday_hour_02_sms_in_mean    0.0
weekday_hour_03_sms_in_mean    0.0
weekday_hour_04_sms_in_mean    0.0
weekday_hour_05_sms_in_mean    0.0
weekday_hour_06_sms_in_mean    0.0
weekday_hour_07_sms_in_mean    0.0
weekday_hour_08_sms_in_mean    0.0
dtype: float64

Rows with any missing: 0.0


We now have 240 temporal-profile features per grid cell. Before clustering, we apply feature selection tailored to unsupervised learning to:
- reduce redundancy and noise (curse of dimensionality),
- improve clustering stability,
- keep features interpretable for stakeholders.

We will proceed in stages:
1) variance screening (identify near-constant features),
2) correlation filtering (remove highly redundant features),
3) optional PCA (only if we decide interpretability can be preserved).
All decisions will be documented in `08_FEATURE_SELECTION.md`.

In [8]:
# compute variance stats and flag low-variance features (if any)

# Separate ID from features
feature_cols = [c for c in X.columns if c != "square_id"]
X_feat = X[feature_cols]

variances = X_feat.var()
low_var = variances[variances < 1e-6].sort_values()  # threshold just for flagging, not final

print("Number of features:", len(feature_cols))
print("Min variance:", variances.min())
print("Median variance:", variances.median())
print("Low-variance features (var < 1e-6):", len(low_var))

# show a few if any exist
display(low_var.head(20).to_frame("variance"))

Number of features: 240
Min variance: 0.025215547885826965
Median variance: 35.919179388631285
Low-variance features (var < 1e-6): 0


,variance


We now need to choose a correlation threshold for removing redundant features. Since your features are hourly profiles, very high correlations can happen (e.g., weekday_hour_10_call_in_mean vs weekday_hour_10_call_out_mean).

In [11]:
threshold = 0.95 # correlation threshold for dropping features

feature_cols = [c for c in X.columns if c != "square_id"]
X_feat = X[feature_cols]

# Correlation matrix
corr = X_feat.corr().abs()

# Upper triangle mask
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# Columns to drop
to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]

print("Correlation threshold:", threshold)
print("Features before:", X_feat.shape[1])
print("Features to drop:", len(to_drop))
print("Features after:", X_feat.shape[1] - len(to_drop))

# Save list for documentation
drop_list_path = RESULTS_PREP_DIR / f"correlated_features_to_drop_thr_{threshold}.csv"
pd.Series(to_drop, name="feature").to_csv(drop_list_path, index=False)
print("Saved drop list:", drop_list_path.resolve())

# Create reduced feature table
X_reduced = pd.concat([X[["square_id"]], X_feat.drop(columns=to_drop)], axis=1)
X_reduced.head()

Correlation threshold: 0.95
Features before: 240
Features to drop: 208
Features after: 32
Saved drop list: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\correlated_features_to_drop_thr_0.95.csv


,square_id,weekday_hour_00_sms_in_mean,weekday_hour_05_sms_in_mean,weekday_hour_06_sms_in_mean,weekend_hour_00_sms_in_mean,weekend_hour_04_sms_in_mean,weekend_hour_05_sms_in_mean,weekend_hour_06_sms_in_mean,weekday_hour_00_sms_out_mean,weekday_hour_06_sms_out_mean,...,weekend_hour_03_call_in_mean,weekend_hour_04_call_in_mean,weekday_hour_00_call_out_mean,weekday_hour_01_call_out_mean,weekday_hour_04_call_out_mean,weekday_hour_05_call_out_mean,weekend_hour_00_call_out_mean,weekend_hour_01_call_out_mean,weekend_hour_04_call_out_mean,weekend_hour_05_call_out_mean
0,1,0.190282,0.117830,0.389212,0.192542,0.030344,0.060287,0.128907,0.156241,0.256934,...,0.005250,0.004794,0.024412,0.008005,0.017220,0.065831,0.037546,0.020957,0.011900,0.031516
1,2,0.192204,0.119678,0.395245,0.194806,0.030850,0.061370,0.131260,0.157375,0.259918,...,0.005339,0.004841,0.024734,0.008009,0.017734,0.067454,0.038205,0.021169,0.012219,0.032364
2,3,0.194250,0.121644,0.401667,0.197215,0.031388,0.062524,0.133765,0.158582,0.263094,...,0.005434,0.004891,0.025077,0.008013,0.018282,0.069182,0.038907,0.021394,0.012559,0.033268
3,4,0.184715,0.112479,0.371735,0.185985,0.028880,0.057148,0.122091,0.152955,0.248292,...,0.004993,0.004658,0.023479,0.007993,0.015730,0.061130,0.035638,0.020342,0.010974,0.029057
4,5,0.167339,0.104840,0.347525,0.168745,0.026729,0.053080,0.114191,0.139368,0.234481,...,0.004558,0.004302,0.021802,0.007469,0.015051,0.057677,0.033155,0.019142,0.010382,0.027518


In [12]:
# Save reduced matrix (32 + square_id)
out_path = RESULTS_PREP_DIR / "features_hourly_weekday_weekend_mean_corrthr_0.95.parquet"
X_reduced.to_parquet(out_path, index=False)

print("Saved reduced feature matrix:", out_path.resolve())
print("Shape:", X_reduced.shape)

Saved reduced feature matrix: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\features_hourly_weekday_weekend_mean_corrthr_0.95.parquet
Shape: (10000, 33)


Lets inspect the remaining features

In [13]:
kept_features = [c for c in X_reduced.columns if c != "square_id"]

print("Kept feature count:", len(kept_features))
print("\nFirst 50 kept features:")
for f in kept_features[:50]:
    print(" -", f)

# How many are weekday vs weekend?
wk = sum(f.startswith("weekday_") for f in kept_features)
we = sum(f.startswith("weekend_") for f in kept_features)
print("\nKept weekday features:", wk)
print("Kept weekend features:", we)

# How many per channel?
channels = ["sms_in", "sms_out", "call_in", "call_out", "internet_traffic"]
for ch in channels:
    n = sum(f"_{ch}_" in f for f in kept_features)
    print(f"Kept for {ch}: {n}")

Kept feature count: 32

First 50 kept features:
 - weekday_hour_00_sms_in_mean
 - weekday_hour_05_sms_in_mean
 - weekday_hour_06_sms_in_mean
 - weekend_hour_00_sms_in_mean
 - weekend_hour_04_sms_in_mean
 - weekend_hour_05_sms_in_mean
 - weekend_hour_06_sms_in_mean
 - weekday_hour_00_sms_out_mean
 - weekday_hour_06_sms_out_mean
 - weekday_hour_07_sms_out_mean
 - weekend_hour_00_sms_out_mean
 - weekend_hour_07_sms_out_mean
 - weekend_hour_08_sms_out_mean
 - weekday_hour_00_call_in_mean
 - weekday_hour_01_call_in_mean
 - weekday_hour_02_call_in_mean
 - weekday_hour_03_call_in_mean
 - weekday_hour_04_call_in_mean
 - weekday_hour_05_call_in_mean
 - weekend_hour_00_call_in_mean
 - weekend_hour_01_call_in_mean
 - weekend_hour_02_call_in_mean
 - weekend_hour_03_call_in_mean
 - weekend_hour_04_call_in_mean
 - weekday_hour_00_call_out_mean
 - weekday_hour_01_call_out_mean
 - weekday_hour_04_call_out_mean
 - weekday_hour_05_call_out_mean
 - weekend_hour_00_call_out_mean
 - weekend_hour_01_call_ou

With the current global correlation filter at 0.95, the kept set is:

- only hours 00–08 (and one 08) → basically night/early morning
- internet_traffic: 0 features kept → completely dropped
- weekday/weekend balance is fine (16/16), but time-of-day coverage is not.

This is not acceptable for our goal (commute/daytime vs nightlife patterns). We should undo this correlation-filtered version and apply a more structured feature selection strategy that preserves:

- coverage across the day
- all channels (especially internet_traffic)
- interpretability

Instead of filtering across everything, we filter redundancy within each channel and regime separately, across the 24 hours. This keeps the “daily rhythm” concept intact and prevents internet traffic from being wiped out. We will try with the same threshold of 0.95 first and see if the channel/regime application will help by itself)

In [16]:
threshold = 0.95 

full_feature_cols = [c for c in X.columns if c != "square_id"]
X_feat = X[full_feature_cols]

def filter_correlated_columns(df_block, thr):
    corr = df_block.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > thr).any()]
    return to_drop

to_drop_all = []

channels = ["sms_in", "sms_out", "call_in", "call_out", "internet_traffic"]
regimes = ["weekday", "weekend"]

for reg in regimes:
    for ch in channels:
        # Select the 24 hourly columns for this (regime, channel)
        cols = [c for c in X_feat.columns if c.startswith(f"{reg}_hour_") and f"_{ch}_" in c]
        block = X_feat[cols]
        block_drop = filter_correlated_columns(block, threshold)
        to_drop_all.extend(block_drop)

to_drop_all = sorted(set(to_drop_all))

print("Threshold:", threshold)
print("Total features before:", X_feat.shape[1])
print("Total features to drop:", len(to_drop_all))
print("Total features after:", X_feat.shape[1] - len(to_drop_all))

# New reduced feature matrix
X_reduced_v2 = pd.concat([X[["square_id"]], X_feat.drop(columns=to_drop_all)], axis=1)

# Save drop list for documentation
drop_list_path = RESULTS_PREP_DIR / f"correlated_features_to_drop_by_channel_regime_thr_{threshold}.csv"
pd.Series(to_drop_all, name="feature").to_csv(drop_list_path, index=False)
print("Saved drop list:", drop_list_path.resolve())

print("X_reduced_v2 shape:", X_reduced_v2.shape)

Threshold: 0.95
Total features before: 240
Total features to drop: 198
Total features after: 42
Saved drop list: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\correlated_features_to_drop_by_channel_regime_thr_0.95.csv
X_reduced_v2 shape: (10000, 43)


In [17]:
kept_features_v2 = [c for c in X_reduced_v2.columns if c != "square_id"]

print("Kept feature count:", len(kept_features_v2))
print("\nFirst 50 kept features:")
for f in kept_features_v2[:50]:
    print(" -", f)

# How many are weekday vs weekend?
wk = sum(f.startswith("weekday_") for f in kept_features_v2)
we = sum(f.startswith("weekend_") for f in kept_features_v2)
print("\nKept weekday features:", wk)
print("Kept weekend features:", we)

# How many per channel?
channels = ["sms_in", "sms_out", "call_in", "call_out", "internet_traffic"]
for ch in channels:
    n = sum(f"_{ch}_" in f for f in kept_features_v2)
    print(f"Kept for {ch}: {n}")

Kept feature count: 42

First 50 kept features:
 - weekday_hour_00_sms_in_mean
 - weekday_hour_05_sms_in_mean
 - weekday_hour_06_sms_in_mean
 - weekend_hour_00_sms_in_mean
 - weekend_hour_04_sms_in_mean
 - weekend_hour_05_sms_in_mean
 - weekend_hour_06_sms_in_mean
 - weekend_hour_07_sms_in_mean
 - weekday_hour_00_sms_out_mean
 - weekday_hour_06_sms_out_mean
 - weekday_hour_07_sms_out_mean
 - weekend_hour_00_sms_out_mean
 - weekend_hour_07_sms_out_mean
 - weekend_hour_08_sms_out_mean
 - weekday_hour_00_call_in_mean
 - weekday_hour_01_call_in_mean
 - weekday_hour_02_call_in_mean
 - weekday_hour_03_call_in_mean
 - weekday_hour_04_call_in_mean
 - weekday_hour_05_call_in_mean
 - weekday_hour_06_call_in_mean
 - weekend_hour_00_call_in_mean
 - weekend_hour_01_call_in_mean
 - weekend_hour_02_call_in_mean
 - weekend_hour_03_call_in_mean
 - weekend_hour_04_call_in_mean
 - weekend_hour_05_call_in_mean
 - weekend_hour_06_call_in_mean
 - weekday_hour_00_call_out_mean
 - weekday_hour_01_call_out_mea

Still daytime is underrepresented. We will try with a higher threshold of 0.975.

In [18]:
threshold = 0.975

full_feature_cols = [c for c in X.columns if c != "square_id"]
X_feat = X[full_feature_cols]

def filter_correlated_columns(df_block, thr):
    corr = df_block.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > thr).any()]
    return to_drop

to_drop_all = []

channels = ["sms_in", "sms_out", "call_in", "call_out", "internet_traffic"]
regimes = ["weekday", "weekend"]

for reg in regimes:
    for ch in channels:
        # Select the 24 hourly columns for this (regime, channel)
        cols = [c for c in X_feat.columns if c.startswith(f"{reg}_hour_") and f"_{ch}_" in c]
        block = X_feat[cols]
        block_drop = filter_correlated_columns(block, threshold)
        to_drop_all.extend(block_drop)

to_drop_all = sorted(set(to_drop_all))

print("Threshold:", threshold)
print("Total features before:", X_feat.shape[1])
print("Total features to drop:", len(to_drop_all))
print("Total features after:", X_feat.shape[1] - len(to_drop_all))

# New reduced feature matrix
X_reduced_v3 = pd.concat([X[["square_id"]], X_feat.drop(columns=to_drop_all)], axis=1)

# Save drop list for documentation
drop_list_path = RESULTS_PREP_DIR / f"correlated_features_to_drop_by_channel_regime_thr_{threshold}.csv"
pd.Series(to_drop_all, name="feature").to_csv(drop_list_path, index=False)
print("Saved drop list:", drop_list_path.resolve())

print("X_reduced_v3 shape:", X_reduced_v3.shape)

Threshold: 0.975
Total features before: 240
Total features to drop: 175
Total features after: 65
Saved drop list: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\correlated_features_to_drop_by_channel_regime_thr_0.975.csv
X_reduced_v3 shape: (10000, 66)


In [21]:
kept_features_v3 = [c for c in X_reduced_v3.columns if c != "square_id"]

print("Kept feature count:", len(kept_features_v3))
print("\nFirst 100 kept features:")
for f in kept_features_v3[:100]:
    print(" -", f)

# How many are weekday vs weekend?
wk = sum(f.startswith("weekday_") for f in kept_features_v3)
we = sum(f.startswith("weekend_") for f in kept_features_v3)
print("\nKept weekday features:", wk)
print("Kept weekend features:", we)

# How many per channel?
channels = ["sms_in", "sms_out", "call_in", "call_out", "internet_traffic"]
for ch in channels:
    n = sum(f"_{ch}_" in f for f in kept_features_v3)
    print(f"Kept for {ch}: {n}")

Kept feature count: 65

First 100 kept features:
 - weekday_hour_00_sms_in_mean
 - weekday_hour_01_sms_in_mean
 - weekday_hour_02_sms_in_mean
 - weekday_hour_03_sms_in_mean
 - weekday_hour_04_sms_in_mean
 - weekday_hour_05_sms_in_mean
 - weekday_hour_06_sms_in_mean
 - weekday_hour_07_sms_in_mean
 - weekend_hour_00_sms_in_mean
 - weekend_hour_01_sms_in_mean
 - weekend_hour_02_sms_in_mean
 - weekend_hour_03_sms_in_mean
 - weekend_hour_04_sms_in_mean
 - weekend_hour_05_sms_in_mean
 - weekend_hour_06_sms_in_mean
 - weekend_hour_07_sms_in_mean
 - weekend_hour_23_sms_in_mean
 - weekday_hour_00_sms_out_mean
 - weekday_hour_01_sms_out_mean
 - weekday_hour_05_sms_out_mean
 - weekday_hour_06_sms_out_mean
 - weekday_hour_07_sms_out_mean
 - weekday_hour_08_sms_out_mean
 - weekday_hour_18_sms_out_mean
 - weekend_hour_00_sms_out_mean
 - weekend_hour_01_sms_out_mean
 - weekend_hour_06_sms_out_mean
 - weekend_hour_07_sms_out_mean
 - weekend_hour_08_sms_out_mean
 - weekday_hour_00_call_in_mean
 - weekd

Even with this threshold, the daytime is not represented in our features. Therefore I decide to take all 240 columns for now. Why keeping all columns is a good choice here:

- Interpretability: we preserve full daily rhythm (commute peaks, midday activity, evening spikes).
- We have enough data: 10,000 squares vs 240 features is totally workable.
- Redundancy isn’t fatal if we handle scaling/transformations properly and validate stability later.
- Correlation filtering is meant to reduce dimensionality, but in our case it’s harming the semantic structure we care about (time-of-day coverage).

In [22]:
# Final feature matrix for clustering: keep ALL temporal profile features
X_final = X.copy()

print("X_final shape:", X_final.shape)

X_final shape: (10000, 241)


In [23]:
out_path = RESULTS_PREP_DIR / "features_hourly_weekday_weekend_mean_FULL.parquet"
X_final.to_parquet(out_path, index=False)

print("Saved:", out_path.resolve())

Saved: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\features_hourly_weekday_weekend_mean_FULL.parquet


# Preprocessing

Preprocessing choices can materially change clustering outcomes. Rather than assuming a single “correct” transformation, we create **two defensible variants** and compare them later in modeling using stability and interpretability checks:

- **Variant A (baseline):** StandardScaler on the full 240-feature matrix  
- **Variant B (skew-robust):** log1p transform (nonnegative heavy-tailed features) + StandardScaler

Both variants will be saved to `results/preprocessing/` so that `04_model_building.ipynb` can load and compare them consistently.

In [25]:
# Use full feature set decided earlier
X_final = X.copy()

id_col = "square_id"
feature_cols = [c for c in X_final.columns if c != id_col]

# Matrix forms
X_raw = X_final[feature_cols].values

# log1p is valid because all features are nonnegative means
X_log = np.log1p(X_raw)

In [27]:
# Fit scalers and transform

scaler_raw = StandardScaler()
scaler_log = StandardScaler()

Z_raw = scaler_raw.fit_transform(X_raw)
Z_log = scaler_log.fit_transform(X_log)

# Save scalers (so modeling is reproducible / consistent)
scaler_raw_path = RESULTS_PREP_DIR / "scaler_variant_A_standard.joblib"
scaler_log_path = RESULTS_PREP_DIR / "scaler_variant_B_log1p_standard.joblib"

joblib.dump(scaler_raw, scaler_raw_path)
joblib.dump(scaler_log, scaler_log_path)

print("Saved scaler A:", scaler_raw_path.resolve())
print("Saved scaler B:", scaler_log_path.resolve())
print("Z_raw shape:", Z_raw.shape, "| Z_log shape:", Z_log.shape)

Saved scaler A: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\scaler_variant_A_standard.joblib
Saved scaler B: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\scaler_variant_B_log1p_standard.joblib
Z_raw shape: (10000, 240) | Z_log shape: (10000, 240)


In [28]:
# Save as DataFrames for easier downstream use
Z_raw_df = pd.DataFrame(Z_raw, columns=feature_cols)
Z_raw_df.insert(0, "square_id", X_final["square_id"].values)

Z_log_df = pd.DataFrame(Z_log, columns=feature_cols)
Z_log_df.insert(0, "square_id", X_final["square_id"].values)

out_A = RESULTS_PREP_DIR / "features_FULL_standardscaled.parquet"
out_B = RESULTS_PREP_DIR / "features_FULL_log1p_standardscaled.parquet"

Z_raw_df.to_parquet(out_A, index=False)
Z_log_df.to_parquet(out_B, index=False)

print("Saved Variant A:", out_A.resolve())
print("Saved Variant B:", out_B.resolve())
print("Variant A shape:", Z_raw_df.shape, "| Variant B shape:", Z_log_df.shape)

Saved Variant A: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\features_FULL_standardscaled.parquet
Saved Variant B: C:\Users\Lu\OneDrive\ToU\chl_Clustering\results\preprocessing\features_FULL_log1p_standardscaled.parquet
Variant A shape: (10000, 241) | Variant B shape: (10000, 241)


In [29]:
# Samity checks 

def sanity_stats(Z_df, name):
    feats = Z_df.drop(columns=["square_id"])
    print(f"\n{name}")
    print("Mean (avg over cols):", feats.mean().mean())
    print("Std  (avg over cols):", feats.std(ddof=0).mean())

sanity_stats(Z_raw_df, "Variant A: StandardScaler(raw)")
sanity_stats(Z_log_df, "Variant B: StandardScaler(log1p)")


Variant A: StandardScaler(raw)
Mean (avg over cols): -2.3566334069376658e-18
Std  (avg over cols): 1.0

Variant B: StandardScaler(log1p)
Mean (avg over cols): 3.6948222259525214e-18
Std  (avg over cols): 1.0
